In [116]:
import sys
import subprocess
from bs4 import BeautifulSoup

In [117]:
name_map = {"Adidas": "Adidas+AG",
            "Nike": "Nike+Inc"
            }

companies = name_map.keys()

In [118]:
site = "https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data"

In [119]:
def get_html(url, selector=None, wait=False):

    cmd = [sys.executable, "webscrape.py", "--url", url]

    if selector:
        cmd += ["--selector", selector]

    if wait:
        cmd += ["--wait"]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("webscrape.py failed")

    with open("page.html", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    return soup



In [120]:
ESG_data = {}
def build_ESG_data(ESG_data, company_name):
    full_site_link = site + "?esg=" + name_map[company_name]
    print(full_site_link)
    soup = get_html(
    full_site_link,
    selector="div.EsgRatings.EsgRatings--rating",
    wait=True
    )
    ESG_container = soup.select_one("div.EsgRatings.EsgRatings--rating")
    print(ESG_container)
    categories = ESG_container.find_all("div", class_="EsgRatings EsgRatings--category")
    ESG_data[company_name] = {}
    for category in categories:
        cat_name = category.find("h5").text
        ESG_data[company_name][cat_name]={}
        sub_cats= category.find_all("div", class_="EsgRatings EsgRatings--item")
        for sub_cat in sub_cats:
            sub_cat_name = sub_cat.find("div").text
            score = sub_cat.find("b").text
            ESG_data[company_name][cat_name][sub_cat_name]=score
    return ESG_data

for company in companies:
    ESG_data = build_ESG_data(ESG_data, company)

print(ESG_data)

https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=Adidas+AG
<div class="EsgRatings EsgRatings--rating"><div class="EsgRatings EsgRatings--category"><div class="EsgRatings EsgRatings--item"><div class="EsgRatings EsgRatings--label"><div class="EsgRatings EsgRatings--color" style="background-color: rgb(0, 144, 147);"></div><h5 class="Typestack Typestack--h5">Environmental<button aria-label="Environment Pillar Score is the weighted average relative rating of a company based on the reported environmental information and the resulting three environmental category scores." class="Tooltip" data-aria-label="Environment Pillar Score is the weighted average relative rating of a company based on the reported environmental information and the resulting three environmental category scores." data-position="top" data-rehydratable="Tooltip" data-show-close-button="false" data-tool-tip-text="Environment Pillar Score is the weighted average relative rating of

In [121]:
import pandas as pd

rows = []

for company, categories in ESG_data.items():
    for category, metrics in categories.items():
        for metric, score in metrics.items():
            rows.append({
                "company": company,
                "category": category,
                "metric": metric,
                "score": score
            })

df = pd.DataFrame(rows)

In [123]:
print(df)

   company       category                         metric score
0   Adidas  Environmental                  Environmental   3.2
1   Adidas  Environmental             Climate Transition     5
2   Adidas  Environmental          Energy & Resource Use     3
3   Adidas  Environmental                   Biodiversity     4
4   Adidas  Environmental                      Water Use     1
5   Adidas  Environmental              Waste & Pollution     2
6   Adidas         Social                         Social   3.8
7   Adidas         Social                Labour Reations     4
8   Adidas         Social                Health & Safety     3
9   Adidas         Social       Human Rights & Community     5
10  Adidas     Governance                     Governance   3.5
11  Adidas     Governance             Board & Management     3
12  Adidas     Governance             Shareholder Rights     4
13  Adidas     Governance      Conduct & Anti-Corruption     4
14  Adidas     Governance  Tax Transperancy & Accountin

In [124]:
df.to_csv("esg_scores.csv", index=False)